#### Phase 1.1 — Load & Structural Inspection

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [3]:
file_path = "../data/Online Retail.xlsx"

df = pd.read_excel(file_path)

In [4]:
df.shape

(541909, 8)

In [5]:
df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom


In [6]:
df.columns.tolist()

['InvoiceNo',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'UnitPrice',
 'CustomerID',
 'Country']

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


#### Phase 1.2 — Data Quality Profiling

In [8]:
# Missing values
df.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [9]:
# Missing value percentages
(df.isna().mean() * 100).round(2)

InvoiceNo       0.00
StockCode       0.00
Description     0.27
Quantity        0.00
InvoiceDate     0.00
UnitPrice       0.00
CustomerID     24.93
Country         0.00
dtype: float64

In [10]:
# Exact duplicate rows
df.duplicated().sum()
#(df.duplicated().mean()*100).round(2)

np.int64(5268)

In [11]:
df[["Quantity", "UnitPrice"]].describe()

,Quantity,UnitPrice
count,541909.000000,541909.000000
mean,9.552250,4.611114
std,218.081158,96.759853
min,-80995.000000,-11062.060000
25%,1.000000,1.250000
50%,3.000000,2.080000
75%,10.000000,4.130000
max,80995.000000,38970.000000


### Initial Data Quality Findings

- CustomerID is missing in 24.93% of transaction rows, which may limit customer-level analysis.
- Description has a relatively small number of missing values (0.27%).
- The dataset contains 5,268 exact duplicate rows that require validation.
- Quantity contains both positive and negative values, requiring investigation of cancellations and returns.
- UnitPrice contains zero/negative and extreme values that require further inspection before revenue calculations.

#### Phase 1.3 — Investigating Transaction Anomalies

What do negative quantities, cancellation invoices, and unusual prices actually represent?

In [12]:
quantity_summary = pd.Series({
    "Positive": (df["Quantity"] > 0).sum(),
    "Zero": (df["Quantity"] == 0).sum(),
    "Negative": (df["Quantity"] < 0).sum()
})

quantity_summary

Positive    531285
Zero             0
Negative     10624
dtype: int64

In [13]:
cancellation_mask = df["InvoiceNo"].astype(str).str.startswith("C")

cancellation_mask.sum()

np.int64(9288)

In [14]:
df.loc[cancellation_mask, "InvoiceNo"].nunique()

3836

In [15]:
pd.crosstab(
    df["Quantity"] < 0,
    cancellation_mask,
    rownames=["Negative Quantity"],
    colnames=["Cancellation Invoice"]
)

Cancellation Invoice,False,True
Negative Quantity,,
False,531285,0
True,1336,9288


All invoice lines explicitly marked as cancellations have negative quantities, but not all negative-quantity records are cancellation invoices.

In [16]:
price_summary = pd.Series({
    "Positive": (df["UnitPrice"] > 0).sum(),
    "Zero": (df["UnitPrice"] == 0).sum(),
    "Negative": (df["UnitPrice"] < 0).sum()
})

price_summary

Positive    539392
Zero          2515
Negative         2
dtype: int64

In [17]:
df[df["UnitPrice"] < 0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


Records with negative UnitPrice correspond to bad-debt accounting adjustments rather than product sales and are therefore excluded from sales KPIs.

In [18]:
df[df["Quantity"].abs() == 80995]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom
540422,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom


#### Phase 1.4 — Investigating 1,336 non-negative and non-C rows 

If some negative quantities are not cancellations, what type of records are they?

In [19]:
non_cancel_negative = df[
    (df["Quantity"] < 0) &
    (~cancellation_mask)
]

non_cancel_negative.head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom
4347,536764,84952C,NaN,-38,2010-12-02 14:42:00,0.0,NaN,United Kingdom
7188,536996,22712,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7189,536997,22028,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7190,536998,85067,NaN,-6,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7192,537000,21414,NaN,-22,2010-12-03 15:32:00,0.0,NaN,United Kingdom
7193,537001,21653,NaN,-6,2010-12-03 15:33:00,0.0,NaN,United Kingdom
7195,537003,85126,NaN,-2,2010-12-03 15:33:00,0.0,NaN,United Kingdom
7196,537004,21814,NaN,-30,2010-12-03 15:34:00,0.0,NaN,United Kingdom
7197,537005,21692,NaN,-70,2010-12-03 15:35:00,0.0,NaN,United Kingdom


In [20]:
non_cancel_negative["Description"].value_counts(dropna=False).head(20)

Description
NaN                           862
check                         120
damages                        45
damaged                        42
?                              41
sold as set on dotcom          20
Damaged                        14
thrown away                     9
Unsaleable, destroyed.          9
??                              7
damages?                        5
wet damaged                     5
ebay                            5
smashed                         4
missing                         3
CHECK                           3
wet pallet                      3
Dotcom sales                    2
reverse 21/5/10 adjustment      2
counted                         2
Name: count, dtype: int64

In [21]:
non_cancel_negative["StockCode"].value_counts().head(20)

StockCode
85175     5
21830     5
85172     4
22719     4
82494L    4
72802C    4
21161     3
22162     3
22034     3
20966     3
22423     3
20892     3
46000S    3
22501     3
22837     3
84631     3
35954     3
84406B    3
22114     3
21621     3
Name: count, dtype: int64

In [22]:
zero_price = df[df["UnitPrice"] == 0]

zero_price.head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom


In [23]:
zero_price["Description"].value_counts(dropna=False).head(20)

Description
NaN                                1454
check                               159
?                                    47
damages                              45
damaged                              43
found                                25
sold as set on dotcom                20
adjustment                           16
Damaged                              14
FRENCH BLUE METAL DOOR SIGN 1         9
thrown away                           9
Unsaleable, destroyed.                9
amazon                                8
FRENCH BLUE METAL DOOR SIGN 8         8
Found                                 8
FRENCH BLUE METAL DOOR SIGN 4         7
FRENCH BLUE METAL DOOR SIGN No        7
OWL DOORSTOP                          7
FRENCH BLUE METAL DOOR SIGN 3         7
RECIPE BOX PANTRY YELLOW DESIGN       7
Name: count, dtype: int64

In [24]:
pd.crosstab(
    zero_price["Quantity"] < 0,
    zero_price["CustomerID"].isna(),
    rownames=["Negative Quantity"],
    colnames=["Missing CustomerID"]
)

Missing CustomerID,False,True
Negative Quantity,,
False,40,1139
True,0,1336


In [25]:
pd.crosstab(
    zero_price["Quantity"] < 0,
    zero_price["CustomerID"].isna(),
    rownames=["Negative Quantity"],
    colnames=["Missing CustomerID"]
)

Missing CustomerID,False,True
Negative Quantity,,
False,40,1139
True,0,1336


In [26]:
missing_description = df[df["Description"].isna()]

missing_description[
    ["InvoiceNo", "StockCode", "Quantity", "UnitPrice", "CustomerID"]
].head(20)

,InvoiceNo,StockCode,Quantity,UnitPrice,CustomerID
622,536414,22139,56,0.0,NaN
1970,536545,21134,1,0.0,NaN
1971,536546,22145,1,0.0,NaN
1972,536547,37509,1,0.0,NaN
1987,536549,85226A,1,0.0,NaN
1988,536550,85044,1,0.0,NaN
2024,536552,20950,1,0.0,NaN
2025,536553,37461,3,0.0,NaN
2026,536554,84670,23,0.0,NaN
2406,536589,21777,-10,0.0,NaN


In [27]:
pd.Series({
    "Total": len(missing_description),
    "Negative Quantity": (missing_description["Quantity"] < 0).sum(),
    "Zero Price": (missing_description["UnitPrice"] == 0).sum(),
    "Missing CustomerID": missing_description["CustomerID"].isna().sum()
})

Total                 1454
Negative Quantity      862
Zero Price            1454
Missing CustomerID    1454
dtype: int64

Negative quantities without cancellation invoice numbers represent operational or inventory adjustments rather than customer sales transactions.

Further investigation showed that negative quantities were not exclusively
customer cancellations. All 1,336 negative-quantity records without a
"C"-prefixed invoice had zero unit prices and missing customer IDs, with
descriptions such as "damages", "thrown away", and "check". These records
appear to represent internal inventory adjustments and should not be treated
as customer sales or returns.

In [28]:
df[
    (df["UnitPrice"] == 0) &
    (df["CustomerID"].notna())
][
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "CustomerID"
    ]
]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,CustomerID
9302,537197,22841,ROUND CAKE TIN VINTAGE GREEN,1,2010-12-05 14:02:00,12647.0
33576,539263,22580,ADVENT CALENDAR GINGHAM SACK,4,2010-12-16 14:36:00,16560.0
40089,539722,22423,REGENCY CAKESTAND 3 TIER,10,2010-12-21 13:45:00,14911.0
47068,540372,22090,PAPER BUNTING RETROSPOT,24,2011-01-06 16:41:00,13081.0
47070,540372,22553,PLASTERS IN TIN SKULLS,24,2011-01-06 16:41:00,13081.0
56674,541109,22168,ORGANISER WOOD ANTIQUE WHITE,1,2011-01-13 15:10:00,15107.0
86789,543599,84535B,FAIRY CAKES NOTEBOOK A6 SIZE,16,2011-02-10 13:08:00,17560.0
130188,547417,22062,CERAMIC BOWL WITH LOVE HEART DESIGN,36,2011-03-23 10:25:00,13239.0
139453,548318,22055,MINI CAKE STAND HANGING STRAWBERY,5,2011-03-30 12:45:00,13113.0
145208,548871,22162,HEART GARLAND RUSTIC PADDED,2,2011-04-04 14:42:00,14410.0


Zero-priced transactions are retained in the cleaned source data but excluded from paid-sales KPIs and product sales rankings because their commercial meaning cannot be reliably determined.

In [29]:
duplicate_rows = df[
    df.duplicated(keep=False)
].sort_values(
    ["InvoiceNo", "StockCode", "InvoiceDate"]
)

duplicate_rows.head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
565,536412,21448,12 DAISY PEGS IN WOOD BOX,2,2010-12-01 11:49:00,1.65,17920.0,United Kingdom
578,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920.0,United Kingdom


In [30]:
len(duplicate_rows)

10147

The raw dataset contained 5,268 exact duplicate rows. Since all available
transaction attributes were identical, only one copy of each duplicated
record was retained to avoid double-counting sales, quantities, and revenue.

## Phase 2 - Data Cleaning

In [31]:
df_clean = df.copy()

#### Step 2.1 — Exact duplicates

In [32]:
rows_before = len(df_clean)

df_clean = df_clean.drop_duplicates().copy()

rows_after = len(df_clean)

print(f"Rows before: {rows_before:,}")
print(f"Rows after:  {rows_after:,}")
print(f"Removed:     {rows_before - rows_after:,}")

Rows before: 541,909
Rows after:  536,641
Removed:     5,268


#### Step 2.2 - Transaction Flags

In [33]:
df_clean["IsCancellation"] = (
    df_clean["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
)

df_clean["IsZeroPrice"] = df_clean["UnitPrice"] == 0

df_clean["IsNegativePrice"] = df_clean["UnitPrice"] < 0

df_clean["HasCustomerID"] = df_clean["CustomerID"].notna()

In [34]:
df_clean["IsOperationalAdjustment"] = (
    (df_clean["Quantity"] < 0) &
    (~df_clean["IsCancellation"]) &
    (df_clean["UnitPrice"] == 0) &
    (df_clean["CustomerID"].isna())
)

In [35]:
df_clean["CustomerID"] = df_clean["CustomerID"].astype("Int64")

In [36]:
df_clean[
    [
        "IsCancellation",
        "IsZeroPrice",
        "IsNegativePrice",
        "IsOperationalAdjustment"
    ]
].sum()

IsCancellation             9251
IsZeroPrice                2510
IsNegativePrice               2
IsOperationalAdjustment    1336
dtype: int64

#### Step 2.3 - Dataset Definition

Which transactions represent actual paid customer purchases?

sales_df

In [37]:
sales_df = df_clean[
    (~df_clean["IsCancellation"]) &
    (df_clean["Quantity"] > 0) &
    (df_clean["UnitPrice"] > 0)
].copy()

cancellation_df

In [38]:
cancellations_df = df_clean[
    (df_clean["IsCancellation"]) &
    (df_clean["Quantity"] < 0) &
    (df_clean["UnitPrice"] > 0)
].copy()

customer_df

In [39]:
customer_df = sales_df[
    sales_df["CustomerID"].notna()
].copy()

In [40]:
sales_df["Revenue"] = (
    sales_df["Quantity"] * sales_df["UnitPrice"]
)

In [46]:
cancellations_df["Revenue"] = (
    cancellations_df["Quantity"] * cancellations_df["UnitPrice"]
)

In [43]:
customer_df = sales_df[
    sales_df["CustomerID"].notna()
].copy()

In [44]:
print(f"Clean rows:         {len(df_clean):,}")
print(f"Paid sales rows:    {len(sales_df):,}")
print(f"Cancellation rows:  {len(cancellations_df):,}")
print(f"Customer sales rows:{len(customer_df):,}")

Clean rows:         536,641
Paid sales rows:    524,878
Cancellation rows:  9,251
Customer sales rows:392,692


In [49]:
revenue_summary = pd.Series({
    "Gross Revenue": sales_df["Revenue"].sum(),
    "Cancellation Value": -cancellations_df["Revenue"].sum(),
    "Net Revenue": (
        sales_df["Revenue"].sum()
        + cancellations_df["Revenue"].sum()
    )
})

revenue_summary.apply(lambda x: f"£{x:,.2f}")

Gross Revenue         £10,642,110.80
Cancellation Value       £893,979.73
Net Revenue            £9,748,131.07
dtype: str

In [48]:
pd.Series({
    "Sales Invoices": sales_df["InvoiceNo"].nunique(),
    "Cancellation Invoices": cancellations_df["InvoiceNo"].nunique(),
    "Customers in Sales Data": customer_df["CustomerID"].nunique()
})

Sales Invoices             19960
Cancellation Invoices       3836
Customers in Sales Data     4338
dtype: int64

In [50]:
pd.Series({
    "Paid Sales Rows": len(sales_df),
    "Paid Sales Rows with CustomerID": sales_df["CustomerID"].notna().sum(),
    "Paid Sales Rows without CustomerID": sales_df["CustomerID"].isna().sum(),
    "CustomerID Coverage (%)": sales_df["CustomerID"].notna().mean() * 100
}).round(2)

Paid Sales Rows                       524878.00
Paid Sales Rows with CustomerID       392692.00
Paid Sales Rows without CustomerID    132186.00
CustomerID Coverage (%)                   74.82
dtype: float64

### Data Cleaning Summary

The raw dataset contained 541,909 transaction lines. After removing 5,268 exact duplicate records, 536,641 rows remained.

Transactions were classified based on their business meaning rather than applying blanket filtering rules. Positive paid transactions were separated from cancellation invoices, zero-priced operational records, and accounting adjustments.

The resulting paid-sales dataset contains 524,878 transaction lines across 19,960 invoices. Customer-level analysis is performed on the subset with identifiable customers, containing 392,692 transaction lines and 4,338 unique customers. Customer IDs are available for 74.82% of paid-sales transaction lines.

Cancellation records were retained separately so that their financial impact can be analyzed rather than being silently discarded.

## Phase 3 — Feature Engineering & KPI Definitions

Business question: 
How should raw transaction lines be transformed into meaningful business metrics?

#### Step 3.1 - Time Features

In [51]:
sales_df["YearMonth"] = (
    sales_df["InvoiceDate"]
    .dt.to_period("M")
)

In [52]:
cancellations_df["YearMonth"] = (
    cancellations_df["InvoiceDate"]
    .dt.to_period("M")
)

In [53]:
customer_df["YearMonth"] = (
    customer_df["InvoiceDate"]
    .dt.to_period("M")
)

In [54]:
sales_df["Weekday"] = sales_df["InvoiceDate"].dt.day_name()
sales_df["Hour"] = sales_df["InvoiceDate"].dt.hour

#### Step 3.2 - Order Value

In [76]:
orders_df = (
    sales_df
    .groupby(
        ["InvoiceNo"],
        as_index=False
    )
    .agg(
        InvoiceDate=("InvoiceDate", "min"),
        OrderValue=("Revenue", "sum"),
        Items=("Quantity", "sum"),
        ProductLines=("StockCode", "count"),
        UniqueProducts=("StockCode", "nunique")
    )
)

In [77]:
len(orders_df)

19960

In [78]:
assert len(orders_df) == sales_df["InvoiceNo"].nunique()

In [73]:
orders_df.head()

,InvoiceNo,OrderValue,Items,ProductLines
0,536365,139.12,40,7
1,536366,22.20,12,2
2,536367,278.73,83,12
3,536368,70.05,15,4
4,536369,17.85,3,1


In [79]:
orders_df["OrderValue"].describe()

count     19960.000000
mean        533.171884
std        1780.412288
min           0.380000
25%         151.695000
50%         303.300000
75%         493.462500
max      168469.600000
Name: OrderValue, dtype: float64

#### Step 3.4 - KPI's definition

AOV ( Average Order Value )

In [68]:
average_order_value = orders_df["OrderValue"].mean()

Repeat Customer Rate

In [72]:
customer_order_counts = (
    customer_df
    .groupby("CustomerID")["InvoiceNo"]
    .nunique()
)

pd.Series({
    "Customers": len(customer_order_counts),
    "One-time Customers": (customer_order_counts == 1).sum(),
    "Repeat Customers": (customer_order_counts > 1).sum(),
    "Repeat Customer Rate (%)": (
        (customer_order_counts > 1).mean() * 100
    )
}).round(2)


Customers                   4338.00
One-time Customers          1493.00
Repeat Customers            2845.00
Repeat Customer Rate (%)      65.58
dtype: float64

A repeat customer is an identified customer with more than one distinct paid-sales invoice during the observation period.